# Chạy pipeline dự đoán LifeMate

Khởi động bằng `bash start_jupyter.sh` để vLLM sẵn sàng, sau đó chạy bốn bước bên dưới theo thứ tự. Notebook dùng chung các hàm dự đoán với `predict.py` và đọc cấu hình từ `config.toml`.

## Bước 1. Cố định seed

Đọc cấu hình và đặt seed giống khi chạy CLI.

In [ ]:
from pathlib import Path

from src.lifemate_config import load_config
from src.zalo_ai.dataset import open_dataset, read_eval_ids
from src.zalo_ai.pipeline import (
    load_prediction_asr,
    predict_samples,
    seed_prediction,
    select_prediction_samples,
)

config = load_config()
seed_prediction()
print("Đã cố định seed cho pipeline dự đoán.")

## Bước 2. Nạp mô hình và tài nguyên

Nạp ASR Gipformer vào bộ nhớ của kernel. Qwen được phục vụ qua vLLM do launcher khởi động.

In [ ]:
load_prediction_asr()
print(f"Endpoint vLLM: {config['prediction']['vllm_url']}")

## Bước 3. Đọc và kiểm tra dữ liệu

Mở đúng một split theo `prediction.split` trong `config.toml`, kiểm tra danh sách ID nếu có và hiển thị số mẫu sẽ chạy.

In [ ]:
data_root = Path(config["paths"]["data_root"])
selected_split = config["prediction"]["split"]
eval_ids = read_eval_ids(config["prediction"]["eval_ids"] or None)

dataset = open_dataset(data_root, selected_split)
samples = select_prediction_samples(dataset, eval_ids)
print(f"File dữ liệu: {dataset.split_dir / (selected_split + '.jsonl')}")
print(f"Split: {selected_split}. Số mẫu được chọn: {len(samples)}.")

## Bước 4. Dự đoán và ghi kết quả

Dùng lại dataset đã mở ở bước 3. Pipeline in tiến độ và thời gian, đồng thời ghi JSONL vào `paths.result_dir` theo tên `jupyter.output_file_name`. Chạy lại bước này sẽ ghi đè file kết quả.

In [ ]:
output_path = Path(config["paths"]["result_dir"]) / config["jupyter"]["output_file_name"]

count = predict_samples(dataset, samples, output_path, eval_ids=eval_ids, overwrite=True)

print(f"Đã ghi {count} mẫu vào {output_path}")